<a href="https://colab.research.google.com/github/Amika1118/DSGP_Group_38/blob/Market-Price-Prediction/Fuel_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%cd /content
!git clone https://github.com/Amika1118/DSGP_Group_38.git
%cd DSGP_Group_38

/content
Cloning into 'DSGP_Group_38'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (77/77), done.
remote: Total 92 (delta 23), reused 47 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 6.05 MiB | 12.59 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/DSGP_Group_38


In [5]:
!git checkout Market-Price-Prediction

Branch 'Market-Price-Prediction' set up to track remote branch 'Market-Price-Prediction' from 'origin'.
Switched to a new branch 'Market-Price-Prediction'


In [6]:
!git config --global user.name "Lasani Layathma"
!git config --global user.email "lasani.20241357@iit.ac.lk"

In [7]:
from getpass import getpass
token = getpass("Enter GitHub token: ")
!git remote set-url origin https://{token}@github.com/Amika1118/DSGP_Group_38.git

Enter GitHub token: ··········


In [34]:
# src/data_preprocessing/prepare_fuel_data.py

import pandas as pd
import os

def prepare_fuel_data(
    input_path="/content/fuel_prices.csv",
    output_path="data/processed/fuel_weekly_cleaned.csv"
):
    # Ensure the output directory exists
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    # Load fuel data
    fuel_df = pd.read_csv(input_path)

    # Keep essential columns
    fuel_df = fuel_df[['date', 'auto_diesel']].copy()
    fuel_df.rename(columns={'auto_diesel': 'Fuel_Price'}, inplace=True)

    # Convert date safely
    fuel_df['date'] = pd.to_datetime(fuel_df['date'], errors='coerce')  # invalid dates -> NaT
    fuel_df = fuel_df.dropna(subset=['date'])  # remove rows with invalid dates

    # Convert to weekly (week ending)
    fuel_df['Week'] = fuel_df['date'].dt.to_period('W').apply(lambda r: r.end_time)

    # Aggregate weekly
    fuel_weekly = fuel_df.groupby('Week')['Fuel_Price'].mean().reset_index()

    # Convert Week to string for CSV
    fuel_weekly['Week'] = fuel_weekly['Week'].dt.strftime('%Y-%m-%d')

    # Handle missing values (forward fill)
    fuel_weekly['Fuel_Price'] = fuel_weekly['Fuel_Price'].ffill()

    # Sort by week
    fuel_weekly = fuel_weekly.sort_values('Week')

    # Save cleaned data
    fuel_weekly.to_csv(output_path, index=False)

    print(" Fuel data cleaned and saved to:", output_path)


if __name__ == "__main__":
    prepare_fuel_data()

 Fuel data cleaned and saved to: data/processed/fuel_weekly_cleaned.csv


In [35]:
!python src/data_preprocessing/prepare_fuel_data.py

python3: can't open file '/content/DSGP_Group_38/src/data_preprocessing/prepare_fuel_data.py': [Errno 2] No such file or directory


In [36]:
import pandas as pd

fuel_weekly = pd.read_csv("data/processed/fuel_weekly_cleaned.csv")
fuel_weekly.head()

,Week,Fuel_Price
0,1990-03-04,9.6
1,1990-08-19,11.0
2,1990-11-11,13.0
3,1990-12-30,11.0
4,1991-01-06,11.0


In [27]:
!git add .
!git commit -m "Add fuel data preprocessing script and generate weekly fuel prices"

[Market-Price-Prediction 5fd9fdb] Add fuel data preprocessing script and generate weekly fuel prices
 1 file changed, 162 insertions(+), 162 deletions(-)
 rewrite data/processed/fuel_weekly_cleaned.csv (99%)


In [28]:
!git push origin Market-Price-Prediction

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 1.18 KiB | 1.18 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Amika1118/DSGP_Group_38.git
   24bc1f1..5fd9fdb  Market-Price-Prediction -> Market-Price-Prediction


In [38]:
import pandas as pd
import os

def merge_retail_fuel_data(
    retail_path="/content/retail_weekly_enhanced.csv",
    fuel_path="/content/fuel_weekly_cleaned (1).csv",
    output_path="data/processed/retail_fuel_weekly_merged.csv"
):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    retail_df = pd.read_csv(retail_path)
    fuel_df = pd.read_csv(fuel_path)

    retail_df["Start_Date"] = pd.to_datetime(retail_df["Start_Date"], errors="coerce")
    fuel_df["Week"] = pd.to_datetime(fuel_df["Week"], errors="coerce")

    retail_df = retail_df.dropna(subset=["Start_Date"])
    fuel_df = fuel_df.dropna(subset=["Week"])

    retail_df = retail_df.sort_values("Start_Date").reset_index(drop=True)
    fuel_df = fuel_df.sort_values("Week").reset_index(drop=True)

    fuel_df = fuel_df.rename(columns={"Week": "Fuel_Effective_Date"})

    merged_df = pd.merge_asof(
        retail_df,
        fuel_df,
        left_on="Start_Date",
        right_on="Fuel_Effective_Date",
        direction="backward"
    )

    merged_df = merged_df.drop(columns=["Fuel_Effective_Date"])

    merged_df.to_csv(output_path, index=False)

    print(" Retail + fuel data merged and saved to:", output_path)


if __name__ == "__main__":
    merge_retail_fuel_data()



 Retail + fuel data merged and saved to: data/processed/retail_fuel_weekly_merged.csv


In [39]:
import pandas as pd

retail_fuel_weekly = pd.read_csv("data/processed/retail_fuel_weekly_merged.csv")
retail_fuel_weekly.head()


,Year,Month,Week_Number,Start_Date,End_Date,Location,Variety,Weekly_Price,Price_Std,Days_In_Week,...,Rolling_Mean_4w,Rolling_Std_4w,Rolling_Mean_13w,Rolling_Std_13w,Rolling_Mean_26w,Rolling_Std_26w,Quarter,Season,Price_Band,Fuel_Price
0,2000,1,1,2000-01-03,2000-01-09,Colombo,BITTER GOURD,37.84,0.18,3,...,37.84,NaN,37.84,NaN,37.84,NaN,1,Winter,Low,13.2
1,2000,1,1,2000-01-03,2000-01-09,Colombo,BRINJALS,36.27,0.10,3,...,36.27,NaN,36.27,NaN,36.27,NaN,1,Winter,Low,13.2
2,2000,1,1,2000-01-03,2000-01-09,Colombo,PUMPKIN,25.28,0.20,3,...,25.28,NaN,25.28,NaN,25.28,NaN,1,Winter,Low,13.2
3,2000,1,1,2000-01-03,2000-01-09,Colombo,TOMATOES,72.82,3.54,3,...,72.82,NaN,72.82,NaN,72.82,NaN,1,Winter,Medium-Low,13.2
4,2000,1,1,2000-01-03,2000-01-09,Colombo,CABBAGE,35.44,0.02,3,...,35.44,NaN,35.44,NaN,35.44,NaN,1,Winter,Low,13.2


In [40]:
!git add .
!git commit -m "Merge weekly fuel prices with retail vegetable prices"


[Market-Price-Prediction ee735ee] Merge weekly fuel prices with retail vegetable prices
 2 files changed, 15570 insertions(+)
 create mode 100644 data/processed/retail_fuel_weekly_merged.csv
 create mode 100644 retail_fuel_weekly_merged.csv


In [41]:
!git push origin Market-Price-Prediction

Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 334.13 KiB | 4.07 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/Amika1118/DSGP_Group_38.git
   5fd9fdb..ee735ee  Market-Price-Prediction -> Market-Price-Prediction
